In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
df_business = session.sql("SELECT * FROM stg_yelp_business").to_pandas()


In [ ]:
df_business.describe()

In [ ]:
df_business.info()

In [ ]:
df_business['CATEGORIES'].head(100)

In [ ]:
df_bus=df_business.copy()

In [ ]:
df_bus = df_bus.assign(
    CATEGORIES = df_business['CATEGORIES'].fillna('').str.split(',')
).explode('CATEGORIES')

df_bus['CATEGORIES'] = df_bus['CATEGORIES'].str.strip()
len(df_bus)

In [ ]:
df_business.head()

In [ ]:
df_bus.head()

In [ ]:
df_business.count()

In [ ]:
df_bus.count()

In [ ]:
## Missing values in the dataset

df_business.isnull().sum()

In [ ]:
SELECT *, COUNT(*) AS cnt
FROM stg_yelp_business
GROUP BY ALL
HAVING cnt > 1
ORDER BY cnt DESC;

In [ ]:
cat_counts = df_bus['CATEGORIES'].value_counts().sort_values(ascending=False)
cat_counts

In [ ]:
cat_counts.__len__()

In [ ]:
df2 = df_bus.copy()
df2['cat'] = df2['CATEGORIES'].str.split(',').apply(lambda x: [c.strip() for c in x])
df2 = df2.explode('cat').rename(columns={'cat':'CATEGORY'})
biz_cat = pd.crosstab(df2['BUSINESS_ID'], df2['CATEGORY'])
print(biz_cat.corr().shape)  # confirms correlation computed

In [ ]:
df = df_bus.copy()

# explode categories
df['cat'] = df['CATEGORIES'].str.split(',')
df = df.explode('cat').rename(columns={'cat':'CATEGORY'})
df['CATEGORY'] = df['CATEGORY'].str.strip()

# self join on business_id to form category pairs
pairs = df.merge(df, on='BUSINESS_ID')
pairs = pairs[pairs['CATEGORY_x'] != pairs['CATEGORY_y']]

# count co-occurrence strength
pair_counts = pairs.groupby(['CATEGORY_x','CATEGORY_y']).size().reset_index(name='count')

# sort strongest pairs
pair_counts = pair_counts.sort_values(by='count', ascending=False)
pair_counts.head(40)

In [ ]:
pair_counts.count()

In [ ]:
strong = pair_counts[pair_counts['count'] > 500]
strong_cats = set(strong['CATEGORY_x']).union(set(strong['CATEGORY_y']))

# show same business in different rows but same category group
df[df['CATEGORY'].isin(strong_cats)][['BUSINESS_ID','NAME','CITY','STARS','CATEGORY']]

In [ ]:
df_business.head()

In [ ]:
df_business.describe()

In [ ]:
df_business.info()

In [ ]:
df_business['CITY'].value_counts().count()

In [ ]:
df_business['STATE'].value_counts().count()

In [ ]:
df_business['OPENED'].value_counts()

In [ ]:
(df_business['OPENED'].value_counts()/df_business.__len__())*100

In [ ]:
print(df_business.groupby('OPENED')['STARS'].mean())


In [ ]:
df_business.groupby('OPENED')['STARS'].describe()


In [ ]:
df_business.groupby('OPENED')['REVIEW_COUNT'].describe()


In [ ]:
df_business.groupby('STATE')['STARS'].mean().sort_values(ascending=False).head(10)


In [ ]:
print(df_business[['STARS','REVIEW_COUNT']].corr())

In [ ]:
print(df_business[['CATEGORIES','STATE','STARS','REVIEW_COUNT']].sort_values('REVIEW_COUNT', ascending=False).head(20))


In [ ]:
print(df_business[['NAME','STATE','STARS','REVIEW_COUNT']].sort_values('REVIEW_COUNT', ascending=False).head(20))


In [ ]:
print(df_bus['CATEGORIES'].value_counts().head(10))


In [ ]:
# Split open/closed
df_open = df_business[df_business['OPENED'] == 1]
df_closed = df_business[df_business['OPENED'] == 0]

# Count categories separately
open_cat = df_open['CATEGORIES'].value_counts()
closed_cat = df_closed['CATEGORIES'].value_counts()

print(open_cat.head(15))
print(closed_cat.head(15))

In [ ]:
# 1. Filter first (choose any condition you want)
df_filtered = df_business[df_business['OPENED'] == 1]  # example: only open businesses
# You can change filter later like: STARS > 3, REVIEW_COUNT > 50, STATE='DL', etc.

# 2. Now run correlation only on the filtered numeric dataset
corr = df_filtered[['STARS','REVIEW_COUNT']].corr()
print(corr)

In [ ]:
# 1. Filter first (choose any condition you want)
df_filter= df_business[df_business['OPENED'] == 0]  # example: only open businesses
# You can change filter later like: STARS > 3, REVIEW_COUNT > 50, STATE='DL', etc.

# 2. Now run correlation only on the filtered numeric dataset
corr = df_filter[['STARS','REVIEW_COUNT']].corr()
print(corr)

In [ ]:
create or replace table joined as
SELECT b.BUSINESS_ID,
       b.NAME,
       b.CITY,
       b.STATE,
       b.STARS,
       b.REVIEW_COUNT,
       b.OPENED,
       r.REVIEW_ID,
       r.REVIEW_STARS,
       r.REVIEW_GIVEN,
       r.SENTIMENT_SCORE
FROM stg_yelp_business AS b
JOIN stg_yelp_reviews AS r
  ON b.BUSINESS_ID = r.BUSINESS_ID

In [ ]:
SELECT CATEGORIES,
       SUM(CASE WHEN OPENED = 1 THEN 1 ELSE 0 END) AS open_count,
       SUM(CASE WHEN OPENED = 0 THEN 1 ELSE 0 END) AS closed_count,
       COUNT(*) AS total,
       open_count * 100.0 / total AS open_pct,
       closed_count * 100.0 / total AS closed_pct
FROM stg_yelp_business
GROUP BY CATEGORIES
ORDER BY total DESC
LIMIT 20;

In [ ]:
SELECT 
  CORR(REVIEW_STARS, SENTIMENT_SCORE) AS corr_reviewStars_sent,
  CORR(REVIEW_STARS, LENGTH(REVIEW_GIVEN)) AS corr_reviewStars_len,
  CORR(SENTIMENT_SCORE, LENGTH(REVIEW_GIVEN)) AS corr_sent_len,
  CORR(REVIEW_COUNT, SENTIMENT_SCORE) AS corr_volume_sent,
  CORR(REVIEW_COUNT, LENGTH(REVIEW_GIVEN)) AS corr_volume_len,

  -- Business STARS correlations (added now)
  CORR(STARS, SENTIMENT_SCORE) AS corr_businessStars_sent,
  CORR(STARS, LENGTH(REVIEW_GIVEN)) AS corr_businessStars_len,
  CORR(STARS, REVIEW_STARS) AS corr_businessStars_reviewStars,
  CORR(STARS, REVIEW_COUNT) AS corr_businessStars_volume
FROM joined
WHERE OPENED = 1

In [ ]:
SELECT 
  CORR(REVIEW_STARS, SENTIMENT_SCORE) AS corr_reviewStars_sent,
  CORR(REVIEW_STARS, LENGTH(REVIEW_GIVEN)) AS corr_reviewStars_len,
  CORR(SENTIMENT_SCORE, LENGTH(REVIEW_GIVEN)) AS corr_sent_len,
  CORR(REVIEW_COUNT, SENTIMENT_SCORE) AS corr_volume_sent,
  CORR(REVIEW_COUNT, LENGTH(REVIEW_GIVEN)) AS corr_volume_len,

  -- Business STARS correlations (added now)
  CORR(STARS, SENTIMENT_SCORE) AS corr_businessStars_sent,
  CORR(STARS, LENGTH(REVIEW_GIVEN)) AS corr_businessStars_len,
  CORR(STARS, REVIEW_STARS) AS corr_businessStars_reviewStars,
  CORR(STARS, REVIEW_COUNT) AS corr_businessStars_volume
FROM joined
WHERE OPENED = 0

In [ ]:
df_joined = session.sql("SELECT * FROM joined").to_pandas()


In [ ]:
df_open = df_joined[df_joined['OPENED'] == 1]
corr_open = df_open[['OPENED','STARS','REVIEW_COUNT','REVIEW_STARS','SENTIMENT_SCORE']].assign(
    REVIEW_LENGTH=df_open['REVIEW_GIVEN'].str.len()
).corr()
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(10,6))
sns.heatmap(corr_open,annot=True)

In [ ]:
df_closed = df_joined[df_joined['OPENED'] == 0]
corr_close = df_clo[['OPENED','STARS','REVIEW_COUNT','REVIEW_STARS','SENTIMENT_SCORE']].assign(
    REVIEW_LENGTH=df_open['REVIEW_GIVEN'].str.len()
).corr()
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(10,6))
sns.heatmap(corr_close,annot=True)

In [ ]:
df_open = df_business[df_business['OPENED'] == 1]

plt.figure()
plt.hist(df_open['STARS'])
plt.title("Stars Distribution (Opened Businesses)")
plt.xlabel("Stars")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure()
plt.hist(df_open['REVIEW_COUNT'], bins=40)
plt.title("Review Count Distribution (Opened Businesses)")
plt.xlabel("Review Count")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure()
plt.scatter(df_open['STARS'], df_open['REVIEW_COUNT'])
plt.title("Stars vs Review Count (Opened Businesses)")
plt.xlabel("Stars")
plt.ylabel("Review Count")
plt.show()

In [ ]:
df_closed = df_business[df_business['OPENED'] == 0]

plt.figure()
plt.hist(df_closed['STARS'])
plt.title("Stars Distribution (Closed Businesses)")
plt.xlabel("Stars")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure()
plt.hist(df_closed['REVIEW_COUNT'], bins=40)
plt.title("Review Count Distribution (Closed Businesses)")
plt.xlabel("Review Count")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure()
plt.scatter(df_closed['STARS'], df_closed['REVIEW_COUNT'])
plt.title("Stars vs Review Count (Closed Businesses)")
plt.xlabel("Stars")
plt.ylabel("Review Count")
plt.show()